In [0]:
import base64
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import padding
import hashlib


import base64
import logging
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import padding
import hashlib


# 
ENCRYPTION_SECRET_KEY = "cdp-mdm-default-encryption-key-2026"


class EncryptionUtil:
    """对称加密工具类（AES-256-CBC）"""
    
    def __init__(self, secret_key: str):
        """
        初始化加密工具
        
        Args:
            secret_key: 加密密钥字符串（将通过SHA-256派生为32字节密钥）
        """
        # 使用SHA-256将密钥派生为32字节（AES-256）
        self.key = hashlib.sha256(secret_key.encode('utf-8')).digest()
        self.block_size = 128  # AES块大小为128位
    
    def encrypt(self, plaintext: str) -> str:
        """
        加密字符串
        
        Args:
            plaintext: 明文字符串
            
        Returns:
            Base64编码的密文字符串
        """
        if plaintext is None or plaintext == '':
            return plaintext
        
        try:
            # 生成随机IV（初始化向量）
            iv = hashlib.md5(plaintext.encode('utf-8')).digest()  # 16字节IV
            
            # 填充明文到块大小的倍数
            padder = padding.PKCS7(self.block_size).padder()
            padded_data = padder.update(plaintext.encode('utf-8')) + padder.finalize()
            
            # 创建加密器并加密
            cipher = Cipher(
                algorithms.AES(self.key),
                modes.CBC(iv),
                backend=default_backend()
            )
            encryptor = cipher.encryptor()
            ciphertext = encryptor.update(padded_data) + encryptor.finalize()
            
            # 将IV和密文拼接后进行Base64编码
            result = base64.b64encode(iv + ciphertext).decode('utf-8')
            
            return result
            
        except Exception as e:
            print(f"Encryption failed: {str(e)}")
            raise
    
    def decrypt(self, ciphertext: str) -> str:
        """
        解密字符串
        
        Args:
            ciphertext: Base64编码的密文字符串
            
        Returns:
            明文字符串
        """
        if ciphertext is None or ciphertext == '':
            return ciphertext
        
        try:
            # Base64解码
            encrypted_data = base64.b64decode(ciphertext.encode('utf-8'))
            
            # 提取IV（前16字节）和密文
            iv = encrypted_data[:16]
            actual_ciphertext = encrypted_data[16:]
            
            # 创建解密器并解密
            cipher = Cipher(
                algorithms.AES(self.key),
                modes.CBC(iv),
                backend=default_backend()
            )
            decryptor = cipher.decryptor()
            padded_plaintext = decryptor.update(actual_ciphertext) + decryptor.finalize()
            
            # 去除填充
            unpadder = padding.PKCS7(self.block_size).unpadder()
            plaintext = unpadder.update(padded_plaintext) + unpadder.finalize()
            
            return plaintext.decode('utf-8')
            
        except Exception as e:
            print(f"Decryption failed: {str(e)}")
            raise


# 全局加密工具实例（延迟初始化）
_encryption_util = None


def get_encryption_util() -> EncryptionUtil:
    """
    获取加密工具实例（单例模式）
    
    Returns:
        EncryptionUtil实例
    """
    global _encryption_util
    
    if _encryption_util is None:
        _encryption_util = EncryptionUtil(ENCRYPTION_SECRET_KEY)
    
    return _encryption_util


def encrypt_phone_number(phone_number: str) -> str:
    """
    加密手机号
    
    Args:
        phone_number: 明文手机号
        
    Returns:
        加密后的手机号
    """
    return get_encryption_util().encrypt(phone_number)


def decrypt_phone_number(encrypted_phone_number: str) -> str:
    """
    解密手机号
    
    Args:
        encrypted_phone_number: 加密的手机号
        
    Returns:
        明文手机号
    """
    return get_encryption_util().decrypt(encrypted_phone_number)


In [0]:
"""
历史数据同步脚本。

从 MySQL 源库 A 同步 rmarket、rcountry、ssourcephoneapiresponse 到 MySQL 目标库 B。
同步过程只清空并重写目标表数据，不创建、删除或修改目标表结构。
"""


from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Tuple

import pymysql


# "mdmmasterfinal.rmarket", "mdmmasterfinal.rcountry", "hkg_mdmlanding.ssourcephoneapiresponse"
TABLES = ["ssourcephoneapiresponse"]
PHONE_TABLE = "ssourcephoneapiresponse"
PHONE_COLUMN = "scpa_phonenumber"
DEFAULT_BATCH_SIZE = 1000

# 源库 连接配置：执行脚本前直接在这里修改为历史数据所在 MySQL 库。
# todo 采用HKG的库
SOURCE_DB_CONFIG = {
    "host": "mysqlflex-ap-southeastasia-prod-cepa-talend-02.mysql.database.azure.com",
    "port": 3306,
    "user": "talend_mdm_read_only@mysql-ap-southeastasia-prod-cepa-talend-02",
    "password": "Q4&4c9JpMSb5A=xa",
    "database": "hkg_mdmlanding",
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "autocommit": False,
}

# 目标库 连接配置：执行脚本前直接在这里修改为需要写入的 MySQL 库。
TARGET_DB_CONFIG = {
    "host": "mysqlflex-ap-southeastasia-prod-cepa-svc-02.mysql.database.azure.com",
    "port": 3306,
    "user": "spark@mysqlflex-ap-southeastasia-prod-cepa-svc-02",
    "password": "tUCdi8bKEML833W!",
    "database": "cdp_mdm",
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "autocommit": False,
}



def read_int_env(name: str, default_value: int) -> int:
    """读取整数环境变量，便于端口、批量大小等参数复用。"""
    value = os.getenv(name)
    return int(value) if value else default_value


def build_db_config(prefix: str) -> Dict[str, Any]:
    """按前缀读取脚本顶部的数据库连接配置，避免源库和目标库配置互相覆盖。"""
    if prefix == "SOURCE":
        return dict(SOURCE_DB_CONFIG)
    if prefix == "TARGET":
        return dict(TARGET_DB_CONFIG)
    raise ValueError(f"Unsupported database config prefix: {prefix}")


def connect_mysql(config: Dict[str, Any]) -> pymysql.connections.Connection:
    """创建 MySQL 连接；非本地地址沿用项目现有 SSL 策略。"""
    connection_config = dict(config)
    if connection_config["host"] != "localhost":
        connection_config["ssl"] = {"ssl_disabled": False}
    return pymysql.connect(**connection_config)


def quote_identifier(identifier: str) -> str:
    """转义 MySQL 标识符，表名和字段名仅来自固定列表或 information_schema。"""
    return "`" + identifier.replace("`", "``") + "`"


def get_insertable_columns(
    connection: pymysql.connections.Connection,
    database: str,
    table_name: str,
) -> List[str]:
    """读取目标表可写字段；以目标 schema 为准，跳过生成列，避免覆盖或改写表结构。"""
    sql = """
        SELECT COLUMN_NAME, EXTRA
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s
        ORDER BY ORDINAL_POSITION
    """
    with connection.cursor() as cursor:
        cursor.execute(sql, (database, table_name))
        rows = cursor.fetchall()

    if not rows:
        raise RuntimeError(f"目标库中不存在表 {table_name}")

    return [
        row["COLUMN_NAME"]
        for row in rows
        if "GENERATED" not in (row.get("EXTRA") or "").upper()
    ]


def get_source_columns(
    connection: pymysql.connections.Connection,
    database: str,
    table_name: str,
) -> List[str]:
    """读取源表字段，用于和目标表字段取交集后同步。"""
    sql = """
        SELECT COLUMN_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s
        ORDER BY ORDINAL_POSITION
    """
    with connection.cursor() as cursor:
        cursor.execute(sql, (database, table_name))
        rows = cursor.fetchall()

    if not rows:
        raise RuntimeError(f"源库中不存在表 {table_name}")

    return [row["COLUMN_NAME"] for row in rows]


def resolve_sync_columns(
    source_connection: pymysql.connections.Connection,
    target_connection: pymysql.connections.Connection,
    source_database: str,
    target_database: str,
    table_name: str,
) -> List[str]:
    """使用目标字段顺序取源目标共有字段，保证 INSERT 不依赖源表和目标表字段顺序完全一致。"""
    target_columns = get_insertable_columns(target_connection, target_database, table_name)
    source_columns = set(get_source_columns(source_connection, source_database, table_name))
    columns = [column for column in target_columns if column in source_columns]

    if not columns:
        raise RuntimeError(f"表 {table_name} 在源库和目标库之间没有可同步的共有字段")

    if table_name == PHONE_TABLE and PHONE_COLUMN not in columns:
        raise RuntimeError(f"表 {PHONE_TABLE} 缺少必要字段 {PHONE_COLUMN}")

    return columns


def clear_target_table(connection: pymysql.connections.Connection, table_name: str) -> None:
    """清空目标表数据；TRUNCATE 不可用时回退 DELETE，整个过程不改动 schema。"""
    quoted_table = quote_identifier(table_name)
    try:
        with connection.cursor() as cursor:
            cursor.execute(f"TRUNCATE TABLE {quoted_table}")
        connection.commit()
    except Exception:
        connection.rollback()
        print("TRUNCATE %s 失败，改用 DELETE 清空目标表数据", table_name)
        with connection.cursor() as cursor:
            cursor.execute(f"DELETE FROM {quoted_table}")
        connection.commit()


def fetch_batches(
    connection: pymysql.connections.Connection,
    table_name: str,
    columns: Sequence[str],
    batch_size: int,
) -> Iterable[List[Dict[str, Any]]]:
    """按批读取源表数据，避免历史数据量较大时一次性加载到内存。"""
    column_sql = ", ".join(quote_identifier(column) for column in columns)
    sql = f"SELECT {column_sql} FROM {quote_identifier(table_name)}"

    with connection.cursor() as cursor:
        cursor.execute(sql)
        while True:
            rows = cursor.fetchmany(batch_size)
            if not rows:
                break
            yield rows


def normalize_phone_value(value: Any) -> Any:
    """同步手机号字段时保留空值，其余值转字符串后调用项目加密方法。"""
    if value is None or value == "":
        return value
    return encrypt_phone_number(str(value))


def build_insert_rows(
    table_name: str,
    columns: Sequence[str],
    rows: Sequence[Dict[str, Any]],
) -> List[Tuple[Any, ...]]:
    """将字典行转换为 INSERT 参数；手机号表在这里完成写入前加密覆盖。"""
    insert_rows = []
    for row in rows:
        values = []
        for column in columns:
            value = row[column]
            if table_name == PHONE_TABLE and column == PHONE_COLUMN:
                value = normalize_phone_value(value)
            values.append(value)
        insert_rows.append(tuple(values))
    return insert_rows


def insert_batch(
    connection: pymysql.connections.Connection,
    table_name: str,
    columns: Sequence[str],
    rows: Sequence[Tuple[Any, ...]],
) -> None:
    """批量写入目标表，仅写入目标表已经存在且源表也存在的字段。"""
    if not rows:
        return

    column_sql = ", ".join(quote_identifier(column) for column in columns)
    placeholders = ", ".join(["%s"] * len(columns))
    sql = f"INSERT INTO {quote_identifier(table_name)} ({column_sql}) VALUES ({placeholders})"

    with connection.cursor() as cursor:
        cursor.executemany(sql, rows)


def sync_table(
    source_connection: pymysql.connections.Connection,
    target_connection: pymysql.connections.Connection,
    source_database: str,
    target_database: str,
    table_name: str,
    batch_size: int,
    dry_run: bool,
) -> int:
    """同步单表：解析字段、清空目标数据、分批读取源数据并写入目标。"""
    columns = resolve_sync_columns(
        source_connection,
        target_connection,
        source_database,
        target_database,
        table_name,
    )
    print("开始同步表 %s，字段数 %s", table_name, len(columns))

    if dry_run:
        print("dry-run: 跳过清空和写入目标表 %s", table_name)
        return 0

    clear_target_table(target_connection, table_name)

    total_count = 0
    try:
        for source_rows in fetch_batches(source_connection, table_name, columns, batch_size):
            insert_rows = build_insert_rows(table_name, columns, source_rows)
            insert_batch(target_connection, table_name, columns, insert_rows)
            target_connection.commit()
            total_count += len(insert_rows)
            print("表 %s 已同步 %s 行", table_name, total_count)
    except Exception:
        target_connection.rollback()
        raise

    print("完成同步表 %s，总行数 %s", table_name, total_count)
    return total_count




def main() -> None:
    """脚本入口：分别连接源库和目标库，按表执行同步并输出汇总。"""
    source_config = build_db_config("SOURCE")
    target_config = build_db_config("TARGET")

    print(f"ENCRYPTION_SECRET_KEY: {ENCRYPTION_SECRET_KEY}")

    source_connection = connect_mysql(source_config)
    target_connection = connect_mysql(target_config)
    try:
        summary = {}
        for table_name in TABLES:
            summary[table_name] = sync_table(
                source_connection,
                target_connection,
                source_config["database"],
                target_config["database"],
                table_name,
                1000,
                False,
            )
        print("历史数据同步完成：%s", summary)
    finally:
        source_connection.close()
        target_connection.close()



    

In [0]:
main()

In [0]:
MYSQL_DRIVER = "com.mysql.cj.jdbc.Driver"

def build_jdbc_url(mysql_host: str, database: str, mysql_use_ssl:bool ) -> str:
    ssl_str = "&useSSL=true&enabledTLSProtocols=TLSv1.2" if mysql_use_ssl else ""

    return (
        f"jdbc:mysql://{mysql_host}:3306/{database}"
        f"?serverTimezone=UTC&rewriteBatchedStatements=true&useUnicode=true&characterEncoding=UTF-8&zeroDateTimeBehavior=CONVERT_TO_NULL&useCompression=true{ssl_str}"
    )

def get_source_mysql(sql_str: str):
    mysql_url = build_jdbc_url(SOURCE_DB_CONFIG["host"], "mdmmasterlanding", True)

    mysql_properties = {
        "user": SOURCE_DB_CONFIG["user"],
        "password": SOURCE_DB_CONFIG["password"],
        "driver": MYSQL_DRIVER,
    }
     
    return spark.read.jdbc(
        url=mysql_url,
        table=sql_str,
        properties=mysql_properties,
    )


def get_target_mysql(sql_str: str):
    mysql_url = build_jdbc_url(TARGET_DB_CONFIG["host"], "cdp_mdm", True)

    mysql_properties = {
        "user": TARGET_DB_CONFIG["user"],
        "password": TARGET_DB_CONFIG["password"],
        "driver": MYSQL_DRIVER,
    }
     
    return spark.read.jdbc(
        url=mysql_url,
        table=sql_str,
        properties=mysql_properties,
    )

# # "mdmmasterfinal.rmarket", "mdmmasterfinal.rcountry", "hkg_mdmlanding.ssourcephoneapiresponse"

In [0]:
display(get_source_mysql("(select * from mdmmasterfinal.rmarket) as t"))

In [0]:
display(get_target_mysql("(select * from cdp_mdm.rmarket) as t"))

In [0]:
display(get_source_mysql("(select * from mdmmasterfinal.rcountry) as t"))

In [0]:
display(get_target_mysql("(select * from cdp_mdm.rcountry) as t")) 

In [0]:
display(get_source_mysql("(select * from hkg_mdmlanding.ssourcephoneapiresponse) as t")) 

In [0]:
print(decrypt_phone_number("UHMRkknIJD+Ve7dZboyTtWIZ7oQmOeXuxBqfwQpY5+U="))
# 13001028396
# 13001028396

In [0]:
display(get_target_mysql("(select * from cdp_mdm.ssourcephoneapiresponse) as t"))  



In [0]:
display(get_source_mysql("(select count(*) from hkg_mdmlanding.ssourcephoneapiresponse) as t")) 

In [0]:
display(get_target_mysql("(select count(*) from cdp_mdm.ssourcephoneapiresponse) as t"))  

In [0]:
get_source_mysql("(select * from hkg_mdmlanding.ssourcephoneapiresponse) as t").createOrReplaceGlobalTempView("source_ssourcephoneapiresponse")

In [0]:
%sql

WITH kv AS (
  SELECT pos + 1 AS ord, x.Column_Name, x.is_null
  FROM global_temp.source_ssourcephoneapiresponse
  LATERAL VIEW posexplode(array(
    named_struct('Column_Name','scpa_id','is_null', `scpa_id` IS NULL),
    named_struct('Column_Name','scpa_phonenumber','is_null', `scpa_phonenumber` IS NULL),
    named_struct('Column_Name','scpa_southchinese_flag','is_null', `scpa_southchinese_flag` IS NULL),
    named_struct('Column_Name','scpa_derived_province','is_null', `scpa_derived_province` IS NULL),
    named_struct('Column_Name','scpa_derived_city','is_null', `scpa_derived_city` IS NULL)
  )) pe AS pos, x
)
SELECT
  Column_Name,
  SUM(CASE WHEN is_null = false THEN 1 ELSE 0 END) AS With_Value,
  SUM(CASE WHEN is_null = true THEN 1 ELSE 0 END) AS null_Value
FROM kv
GROUP BY ord, Column_Name
ORDER BY ord;

In [0]:
get_target_mysql("(select * from cdp_mdm.ssourcephoneapiresponse) as t").createOrReplaceGlobalTempView("target_ssourcephoneapiresponse")

In [0]:
%sql

        

WITH kv AS (
  SELECT pos + 1 AS ord, x.Column_Name, x.is_null
  FROM global_temp.target_ssourcephoneapiresponse
  LATERAL VIEW posexplode(array(
    named_struct('Column_Name','scpa_id','is_null', `scpa_id` IS NULL),
    named_struct('Column_Name','scpa_phonenumber','is_null', `scpa_phonenumber` IS NULL),
    named_struct('Column_Name','scpa_southchinese_flag','is_null', `scpa_southchinese_flag` IS NULL),
    named_struct('Column_Name','scpa_derived_province','is_null', `scpa_derived_province` IS NULL),
    named_struct('Column_Name','scpa_derived_city','is_null', `scpa_derived_city` IS NULL)
  )) pe AS pos, x
)
SELECT
  Column_Name,
  SUM(CASE WHEN is_null = false THEN 1 ELSE 0 END) AS With_Value,
  SUM(CASE WHEN is_null = true THEN 1 ELSE 0 END) AS null_Value
FROM kv
GROUP BY ord, Column_Name
ORDER BY ord;